In [1]:
import pandas as pd 

matches = pd.read_csv("../data/matches_1930_2022.csv")
rankings = pd.read_csv("../data/fifa_ranking_2026-06-08.csv")

In [2]:
TEAM_MAPPING = {
    "United States": "USA",
    "Czech Republic": "Czechia",
    "West Germany": "Germany",
    "Germany DR": "Germany",
    "Soviet Union": "Russia",
    "Yugoslavia": "Serbia",
    "FR Yugoslavia": "Serbia",
    "Serbia and Montenegro": "Serbia",
    "Zaire": "Congo DR",
    "Dutch East Indies": "Indonesia",
    "Czechoslovakia": "Czechia"
}

In [3]:
matches["home_team"] = matches["home_team"].replace(TEAM_MAPPING)
matches["away_team"] = matches["away_team"].replace(TEAM_MAPPING)

In [4]:
ranking_lookup = rankings[
    ["team", "rank", "points", "rated_matches"]
].copy()

ranking_lookup.head()

,team,rank,points,rated_matches
0,Argentina,1,1876.118331,59
1,Spain,2,1873.013187,56
2,France,3,1869.428449,57
3,England,4,1827.048678,57
4,Portugal,5,1766.177547,56


In [5]:
home_rankings = ranking_lookup.add_prefix("home_")
away_rankings = ranking_lookup.add_prefix("away_")

In [6]:
df = matches.merge(
    home_rankings,
    left_on="home_team",
    right_on="home_team",
    how="left"
)

df = df.merge(
    away_rankings,
    left_on="away_team",
    right_on="away_team",
    how="left"
)

In [7]:
df.shape

(964, 50)

In [8]:
df[
    [
        "home_team",
        "away_team",
        "home_rank",
        "away_rank",
        "home_points",
        "away_points"
    ]
].head()

,home_team,away_team,home_rank,away_rank,home_points,away_points
0,Argentina,France,1,3,1876.118331,1869.428449
1,Croatia,Morocco,11,7,1714.865572,1755.100232
2,France,Morocco,3,7,1869.428449,1755.100232
3,Argentina,Croatia,1,11,1876.118331,1714.865572
4,Morocco,Portugal,7,5,1755.100232,1766.177547


In [9]:
df["rank_difference"] = (
    df["home_rank"] - df["away_rank"]
)

df["points_difference"] = (
    df["home_points"] - df["away_points"]
)

df["experience_difference"] = (
    df["home_rated_matches"] -
    df["away_rated_matches"]
)

In [10]:
df[
    [
        "rank_difference",
        "points_difference",
        "experience_difference"
    ]
].describe()

,rank_difference,points_difference,experience_difference
count,964.000000,964.000000,964.000000
mean,-6.657676,47.802634,-1.066390
std,33.192648,227.523363,14.232083
min,-130.000000,-722.724831,-51.000000
25%,-26.000000,-116.343010,-6.000000
50%,-5.000000,49.810735,0.000000
75%,11.000000,214.998814,4.000000
max,119.000000,762.953862,53.000000


## Phase 4 - Create Winner Labels

In [11]:
def get_winner(row):

    if row["home_score"] > row["away_score"]:
        return 1

    elif row["home_score"] < row["away_score"]:
        return 0

    else:
        return 2

In [12]:
df["winner"] = df.apply(
    get_winner,
    axis=1
)

In [13]:
df["winner"].value_counts()

winner
1    532
0    218
2    214
Name: count, dtype: int64

In [14]:
df["winner"].value_counts(normalize=True) * 100

winner
1    55.186722
0    22.614108
2    22.199170
Name: proportion, dtype: float64

In [15]:
df[
    [
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner"
    ]
].head(10)

,home_team,away_team,home_score,away_score,winner
0,Argentina,France,3,3,2
1,Croatia,Morocco,2,1,1
2,France,Morocco,2,0,1
3,Argentina,Croatia,3,0,1
4,Morocco,Portugal,1,0,1
5,England,France,1,2,0
6,Croatia,Brazil,1,1,2
7,Netherlands,Argentina,2,2,2
8,Morocco,Spain,0,0,2
9,Portugal,Switzerland,6,1,1


## Phase 5 — First Model 

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [17]:
features = [
    "rank_difference",
    "points_difference",
    "experience_difference"
]

X = df[features]

y = df["winner"]

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [19]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [20]:
preds = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    preds
)

print("Accuracy:", accuracy)

Accuracy: 0.5129533678756477


In [21]:
print(
    classification_report(
        y_test,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.37      0.36      0.37        44
           1       0.63      0.73      0.68       106
           2       0.21      0.14      0.17        43

    accuracy                           0.51       193
   macro avg       0.41      0.41      0.40       193
weighted avg       0.48      0.51      0.49       193



In [22]:
df[
    [
        "home_xg",
        "away_xg"
    ]
].isnull().sum()

home_xg    836
away_xg    836
dtype: int64

In [23]:
df[
    [
        "home_xg",
        "away_xg"
    ]
].describe()

,home_xg,away_xg
count,128.000000,128.000000
mean,1.435938,1.192187
std,0.860391,0.809258
min,0.100000,0.000000
25%,0.800000,0.600000
50%,1.300000,1.000000
75%,1.925000,1.525000
max,5.200000,5.700000


In [24]:
df["Round"].value_counts()

Round
Group stage             587
Round of 16              97
Quarter-finals           70
First round              48
Semi-finals              38
First group stage        36
Second round             24
Final                    21
Third-place match        20
Second group stage       12
Final stage               6
Group stage play-off      5
Name: count, dtype: int64

In [25]:
df["Year"].min(), df["Year"].max()

(1930, 2022)

## Phase 5.5 — Advanced Features 

In [26]:
df["home_rank_inverse"] = 1 / df["home_rank"]
df["away_rank_inverse"] = 1 / df["away_rank"]

In [27]:
df["home_is_top10"] = (
    df["home_rank"] <= 10
).astype(int)

df["away_is_top10"] = (
    df["away_rank"] <= 10
).astype(int)

In [28]:
df["same_confederation"] = 0

In [29]:
df["modern_era"] = (
    df["Year"] >= 1998
).astype(int)

In [30]:
round_map = {
    "Group stage":1,
    "First group stage":1,
    "Second group stage":2,
    "Round of 16":3,
    "Quarter-finals":4,
    "Semi-finals":5,
    "Final":6,
    "Third-place match":5,
    "Second round":3,
    "First round":2,
    "Final stage":6
}

df["round_encoded"] = (
    df["Round"]
    .map(round_map)
    .fillna(1)
)

In [31]:
features = [
    "rank_difference",
    "points_difference",
    "experience_difference",
    "home_rank_inverse",
    "away_rank_inverse",
    "home_is_top10",
    "away_is_top10",
    "modern_era",
    "round_encoded"
]

In [34]:
X = df[features]

y = df["winner"]

print(X.shape)
print(y.shape)

(964, 9)
(964,)


In [35]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(771, 9)
(193, 9)


In [44]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [45]:
import joblib

joblib.dump(
    model,
    "../models/winner_model.pkl"
)

print("Model Saved Successfully")

Model Saved Successfully


In [37]:
preds = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    preds
)

print("Accuracy:", accuracy)

Accuracy: 0.5751295336787565


In [38]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.45      0.50      0.47        44
           1       0.67      0.79      0.73       106
           2       0.26      0.12      0.16        43

    accuracy                           0.58       193
   macro avg       0.46      0.47      0.45       193
weighted avg       0.53      0.58      0.54       193



In [46]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance.sort_values(
    by="importance",
    ascending=False
)

,feature,importance
1,points_difference,0.191451
0,rank_difference,0.175766
2,experience_difference,0.172704
3,home_rank_inverse,0.146134
4,away_rank_inverse,0.144397
7,modern_era,0.074193
8,round_encoded,0.070166
6,away_is_top10,0.012852
5,home_is_top10,0.012338


In [43]:
import os

os.makedirs("../models", exist_ok=True)

print("Models folder ready")

Models folder ready


In [47]:
import os

print(
    os.path.exists("../models/winner_model.pkl")
)

True
